# View Purge Behavior Test — purge-view-metadata-on-drop

**Goal:** Document what happens when a view is dropped under each config value,
with confirmed-working MinIO credentials.

This answers the manager's question precisely:
> What happens when you purge a view inside a catalog where
> `polaris.config.purge-view-metadata-on-drop = true` (the default)?

## Test matrix
```
Case 1: purge-view-metadata-on-drop = TRUE  (default)
        → does the view metadata file get deleted from MinIO?
        → does the drop succeed or 403?

Case 2: purge-view-metadata-on-drop = FALSE
        → drop is metadata-only (PostgreSQL), MinIO untouched?

We observe MinIO directly to confirm whether the .metadata.json
view file is actually deleted.
```


In [1]:
import sys
sys.path.insert(0, '.')
from polaris_test_utils import *
import s3fs, time

tok = root_token()
print("✅ root token")

TEST_CATALOG  = "view-purge-test"
TEST_NS       = "vp-ns"
TEST_LOCATION = "s3a://data-catalog-bucket/view-purge-test/"

MINIO_ENDPOINT   = "http://192.168.139.2:9000"
MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "bhEenDzTEAVPKbukhuY0tRY4WjpZXzh0"

fs = s3fs.S3FileSystem(
    key=MINIO_ACCESS_KEY, secret=MINIO_SECRET_KEY,
    use_ssl=False, client_kwargs={"endpoint_url": MINIO_ENDPOINT}
)

def list_view_files(view_name):
    """List MinIO files under a view's location."""
    prefix = f"data-catalog-bucket/view-purge-test/{TEST_NS}/{view_name}/"
    try:
        files = fs.ls(prefix, detail=False)
        return files
    except FileNotFoundError:
        return []


✅ Polaris test utils loaded
   Polaris: http://192.168.139.2:8181  | Realm: POLARIS
   OpenSearch: https://localhost:9200
✅ root token


In [2]:
# Cleanup any leftover from prior runs
print("Cleanup...")
catalogs = requests.get(f"{BASE_MGMT}/catalogs", headers=h(tok)).json().get("catalogs", [])
if any(c["name"] == TEST_CATALOG for c in catalogs):
    # enable purge + grant so we can fully clear
    cat = requests.get(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok)).json()
    merged = {**cat.get("properties", {}),
              "polaris.config.drop-with-purge.enabled": "true",
              "polaris.config.purge-view-metadata-on-drop": "false"}
    requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok),
                 json={"currentEntityVersion": cat["entityVersion"], "properties": merged})
    requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}/catalog-roles/catalog_admin/grants",
                 headers=h(tok), json={"grant": {"type":"catalog","privilege":"CATALOG_MANAGE_CONTENT"}})
    for ns in requests.get(f"{BASE_CAT}/{TEST_CATALOG}/namespaces", headers=h(tok)).json().get("namespaces", []):
        nsn = ns[0] if isinstance(ns, list) else ns
        for v in requests.get(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", []):
            requests.delete(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{nsn}/views/{v['name']}?purgeRequested=true", headers=h(tok))
        requests.delete(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{nsn}", headers=h(tok))
    requests.delete(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}?purgeRequested=true", headers=h(tok))
    print("  removed old test catalog")
print("✅ clean")


Cleanup...
  removed old test catalog
✅ clean


In [3]:
# What's actually left in view-purge-test?
import json as _json
r = requests.get(f"{BASE_MGMT}/catalogs/view-purge-test", headers=h(tok))
print(f"Catalog exists: {r.status_code}")

# List its namespaces and their contents
ns_r = requests.get(f"{BASE_CAT}/view-purge-test/namespaces", headers=h(tok))
print(f"Namespaces: {ns_r.json()}")

for ns in ns_r.json().get("namespaces", []):
    nsn = ns[0] if isinstance(ns, list) else ns
    tabs = requests.get(f"{BASE_CAT}/view-purge-test/namespaces/{nsn}/tables", headers=h(tok)).json()
    views = requests.get(f"{BASE_CAT}/view-purge-test/namespaces/{nsn}/views", headers=h(tok)).json()
    print(f"  {nsn}: tables={tabs.get('identifiers',[])}  views={views.get('identifiers',[])}")

Catalog exists: 404
Namespaces: {'error': {'message': 'Namespace does not exist: ', 'type': 'NoSuchNamespaceException', 'code': 404}}


In [4]:
CATALOG = "view-purge-test"

# 1. Enable purge + grant (flat read, merged PUT)
cat = requests.get(f"{BASE_MGMT}/catalogs/{CATALOG}", headers=h(tok)).json()
if "entityVersion" in cat:
    merged = {**cat.get("properties", {}),
              "polaris.config.drop-with-purge.enabled": "true",
              "polaris.config.purge-view-metadata-on-drop": "false"}
    rp = requests.put(f"{BASE_MGMT}/catalogs/{CATALOG}", headers=h(tok),
                      json={"currentEntityVersion": cat["entityVersion"], "properties": merged})
    print(f"enable purge: {rp.status_code}")
    requests.put(f"{BASE_MGMT}/catalogs/{CATALOG}/catalog-roles/catalog_admin/grants",
                 headers=h(tok), json={"grant":{"type":"catalog","privilege":"CATALOG_MANAGE_CONTENT"}})

# 2. Delete all contents, reporting every failure
for ns in requests.get(f"{BASE_CAT}/{CATALOG}/namespaces", headers=h(tok)).json().get("namespaces", []):
    nsn = ns[0] if isinstance(ns, list) else ns
    for t in requests.get(f"{BASE_CAT}/{CATALOG}/namespaces/{nsn}/tables", headers=h(tok)).json().get("identifiers", []):
        r = requests.delete(f"{BASE_CAT}/{CATALOG}/namespaces/{nsn}/tables/{t['name']}?purgeRequested=true", headers=h(tok))
        print(f"table {nsn}.{t['name']}: {r.status_code}" + ("" if r.status_code in (200,204) else f" {r.text[:120]}"))
    for v in requests.get(f"{BASE_CAT}/{CATALOG}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", []):
        r = requests.delete(f"{BASE_CAT}/{CATALOG}/namespaces/{nsn}/views/{v['name']}?purgeRequested=true", headers=h(tok))
        print(f"view {nsn}.{v['name']}: {r.status_code}" + ("" if r.status_code in (200,204) else f" {r.text[:120]}"))
    r = requests.delete(f"{BASE_CAT}/{CATALOG}/namespaces/{nsn}", headers=h(tok))
    print(f"namespace {nsn}: {r.status_code}" + ("" if r.status_code in (200,204) else f" {r.text[:120]}"))

# 3. Delete catalog
r = requests.delete(f"{BASE_MGMT}/catalogs/{CATALOG}?purgeRequested=true", headers=h(tok))
print(f"catalog: {r.status_code}" + ("" if r.status_code in (200,204) else f" {r.text[:150]}"))

# 4. VERIFY it's actually gone
r = requests.get(f"{BASE_MGMT}/catalogs/{CATALOG}", headers=h(tok))
print(f"\nVerify catalog gone: {r.status_code} ({'✅ gone' if r.status_code==404 else '❌ STILL EXISTS'})")

catalog: 404 {"error":{"message":"TopLevelEntity of type CATALOG does not exist: view-purge-test","type":"NotFoundException","code":404}}

Verify catalog gone: 404 (✅ gone)


In [5]:
# The whole catalog was just purged. Are its MinIO files gone, or orphaned?
import time
print("Immediately after catalog purge:")
files = [f for f in fs.find("data-catalog-bucket/view-purge-test/") if fs.info(f)['size'] > 0]
print(f"  files remaining: {len(files)}")

# Wait for async cleanup, re-check
for wait in [2, 4, 6]:
    time.sleep(2)
    files = [f for f in fs.find("data-catalog-bucket/view-purge-test/") if fs.info(f)['size'] > 0]
    print(f"  +{wait}s: {len(files)} files")
    if len(files) == 0:
        print("  ✅ all files cleaned by async task")
        break

if files:
    print("\n  Remaining files:")
    for f in files[:10]:
        print(f"    {f}")

Immediately after catalog purge:
  files remaining: 2
  +10s: 2 files
  +20s: 2 files
  +30s: 2 files

  Remaining files:
    data-catalog-bucket/view-purge-test/vp-ns/view-true/metadata/00000-b698d300-333c-4748-bc40-53e8c8d16cce.gz.metadata.json
    data-catalog-bucket/view-purge-test/vp-ns/view-true/metadata/00000-ea4b2231-cf0a-48a6-8f6e-2d12ec7160ea.gz.metadata.json


In [6]:
from opensearchpy import OpenSearch
try:
    client = os_client(); INDEX = LOG_INDEX
except Exception:
    client = OpenSearch(hosts=[{"host":"192.168.194.1","port":9200}],
        http_auth=("admin","Str0ngP@ssw0rd123!"),
        use_ssl=True, verify_certs=False, ssl_show_warn=False)
    INDEX = "kube-fb-log"

resp = client.search(index=INDEX, body={
    "query": {"bool": {
        "must": [{"range": {"@timestamp": {"gte": "now-30m"}}},
                 {"match_phrase": {"kubernetes.container_name": "polaris"}}],
        "should": [
            {"match_phrase": {"message": "TableCleanupTaskHandler"}},
            {"match_phrase": {"message": "TaskExecutor"}},
            {"match_phrase": {"message": "cleanup"}},
            {"match_phrase": {"message": "purge"}},
            {"match_phrase": {"message": "Handling task"}},
            {"match_phrase": {"message": "FileIO"}},
        ],
        "minimum_should_match": 1}},
    "sort": [{"@timestamp": {"order": "desc"}}], "size": 40})

hits = resp["hits"]["hits"]
print(f"Found {len(hits)} entries:\n")
for h in hits:
    s = h["_source"]
    print(f"[{s.get('@timestamp','')[:19]}] {s.get('level','?')}")
    print(f"  {s.get('message','')[:250]}\n")

if not hits:
    print("No matches. Try without the container filter, or check if Polaris")
    print("logs are JSON-parsed. Broaden: just match_phrase 'task' on message.")

Found 40 entries:

[2026-06-26T02:54:58] INFO
  192.168.194.1 - root [26/Jun/2026:02:54:58 +0000] "GET /api/management/v1/catalogs/view-purge-test HTTP/1.1" 404 124

[2026-06-26T02:54:58] INFO
  Handling runtimeException TopLevelEntity of type CATALOG does not exist: view-purge-test

[2026-06-26T02:54:58] INFO
  Handling runtimeException TopLevelEntity of type CATALOG does not exist: view-purge-test

[2026-06-26T02:54:58] INFO
  192.168.194.1 - root [26/Jun/2026:02:54:58 +0000] "DELETE /api/management/v1/catalogs/view-purge-test?purgeRequested=true HTTP/1.1" 404 124

[2026-06-26T02:54:58] INFO
  192.168.194.1 - root [26/Jun/2026:02:54:58 +0000] "GET /api/catalog/v1/view-purge-test/namespaces HTTP/1.1" 404 95

[2026-06-26T02:54:58] INFO
  Handling runtimeException TopLevelEntity of type CATALOG does not exist: view-purge-test

[2026-06-26T02:54:58] INFO
  192.168.194.1 - root [26/Jun/2026:02:54:58 +0000] "GET /api/management/v1/catalogs/view-purge-test HTTP/1.1" 404 124

[2026-06-26T02:

In [7]:
import uuid, time
import pyarrow as pa, pyarrow.parquet as pq

# Use the catalog that exists. Confirm which one:
cats = [c["name"] for c in requests.get(f"{BASE_MGMT}/catalogs", headers=h(tok)).json().get("catalogs", [])]
print("Catalogs:", cats)

CAT = "test-error-catalog"   # adjust if needed based on output above
NS  = "vp-ns"

# Ensure namespace exists
requests.post(f"{BASE_CAT}/{CAT}/namespaces", headers=h(tok),
              json={"namespace": [NS], "properties": {}})

# Enable purge + grant
cat = requests.get(f"{BASE_MGMT}/catalogs/{CAT}", headers=h(tok)).json()
merged = {**cat.get("properties", {}),
          "polaris.config.drop-with-purge.enabled": "true"}
requests.put(f"{BASE_MGMT}/catalogs/{CAT}", headers=h(tok),
             json={"currentEntityVersion": cat["entityVersion"], "properties": merged})
requests.put(f"{BASE_MGMT}/catalogs/{CAT}/catalog-roles/catalog_admin/grants",
             headers=h(tok), json={"grant":{"type":"catalog","privilege":"CATALOG_MANAGE_CONTENT"}})

# CREATE a real table
TBL = f"logtest-{uuid.uuid4().hex[:8]}"
loc = f"s3a://data-catalog-bucket/{CAT}/{NS}/{TBL}"
r = requests.post(f"{BASE_CAT}/{CAT}/namespaces/{NS}/tables", headers=h(tok),
    json={"name": TBL, "location": f"{loc}/",
          "schema": {"type":"struct","fields":[{"id":1,"name":"id","type":"long","required":True}]},
          "properties": {"write.format.default":"parquet"}})
print(f"Create {TBL}: {r.status_code}")

# Write a real Parquet file
data = pa.table({"id": pa.array([1,2,3], type=pa.int64())})
pqp = f"{loc}/data/{uuid.uuid4().hex}.parquet".replace("s3a://","s3://")
with fs.open(pqp, "wb") as f:
    pq.write_table(data, f)
print(f"Wrote Parquet: {fs.info(pqp)['size']} bytes")

# Note the request id of the PURGE so we can trace its cleanup task
r = requests.delete(f"{BASE_CAT}/{CAT}/namespaces/{NS}/tables/{TBL}?purgeRequested=true", headers=h(tok))
purge_req_id = get_request_id(r)
print(f"PURGE {TBL}: {r.status_code}, request-id: {purge_req_id}")

print("\nWaiting 10s for async cleanup task to run...")
time.sleep(10)

TypeError: 'dict' object is not callable

In [8]:
# Cleanup task logs since the purge
hits = search_logs_raw({
    "query": {"bool": {
        "must": [
            {"range": {"@timestamp": {"gte": "now-5m"}}},
            {"term": {"kubernetes.container_name.keyword": "benchmarks-polaris"}}],
        "should": [
            {"match_phrase": {"loggerName": "TableCleanupTaskHandler"}},
            {"match_phrase": {"loggerName": "TaskExecutor"}},
            {"match_phrase": {"loggerName": "CatalogUtil"}},
            {"match_phrase": {"loggerName": "S3FileIO"}},
            {"match_phrase": {"loggerName": "FileIOTracker"}},
            {"match_phrase": {"message": "cleanup"}},
            {"match_phrase": {"message": "Handling task"}},
            {"match_phrase": {"message": TBL}}],   # match our specific table
        "minimum_should_match": 1}},
    "sort": [{"@timestamp": {"order": "asc"}}]   # chronological
}, size=50)
print_logs(hits, show_debug_on_empty=False)

NameError: name 'TBL' is not defined

In [9]:
# And the safety net — any ERROR/WARN since the purge
hits = search_logs_raw({
    "query": {"bool": {"must": [
        {"range": {"@timestamp": {"gte": "now-5m"}}},
        {"term": {"kubernetes.container_name.keyword": "benchmarks-polaris"}},
        {"terms": {"level.keyword": ["ERROR", "WARN"]}}]}},
    "sort": [{"@timestamp": {"order": "asc"}}]
}, size=30)
print_logs(hits, show_debug_on_empty=False)

Found 1 log entries:

❌ [2026-06-26T02:54:38.309Z] ERROR
   logger    : org.apache.polaris.service.exception.IcebergExceptionMapper
   requestId : d2499e4f-0c44-4779-9aa0-860d9d3a8f33_0000000000000000835
   realmId   : POLARIS
   message   : Unhandled exception returning INTERNAL_SERVER_ERROR



In [10]:
# Get the FULL failure detail — the WARN truncates the cause.
# Look for the exception/stacktrace around the failed task, and the full message.
hits = search_logs_raw({
    "query": {"bool": {
        "must": [
            {"range": {"@timestamp": {"gte": "now-10m"}}},
            {"term": {"kubernetes.container_name.keyword": "benchmarks-polaris"}}],
        "should": [
            {"match_phrase": {"message": "9008045721606047915"}},   # the task id
            {"match_phrase": {"message": "Failed to handle task"}},
            {"match_phrase": {"loggerName": "TableCleanupTaskHandler"}},
            {"wildcard": {"message": "*xception*"}},
            {"wildcard": {"message": "*rror*"}},
            {"match_phrase": {"loggerName": "TaskExecutorImpl"}}],
        "minimum_should_match": 1}},
    "sort": [{"@timestamp": {"order": "asc"}}]
}, size=60)

# Print FULL messages (not truncated) to catch the stack trace
print(f"Found {len(hits)} entries:\n")
for h in hits:
    s = h["_source"]
    ts = s.get("@timestamp","")[:23]
    lvl = s.get("level","")
    lg = s.get("loggerName","")
    msg = s.get("message","")   # FULL, not truncated
    print(f"[{ts}] {lvl} | {lg}")
    print(f"  {msg}")
    print()

Found 35 entries:

[2026-06-26T02:46:05.791] INFO | org.apache.polaris.service.exception.IcebergExceptionMapper
  Handling runtimeException Property 'polaris.config.drop-with-purge-enabled' matches reserved prefix 'polaris.'

[2026-06-26T02:46:05.799] INFO | org.apache.polaris.service.exception.IcebergExceptionMapper
  Handling runtimeException Failed to write to grant records due to Failed due to 'ERROR: duplicate key value violates unique constraint "grant_records_pkey"
  Detail: Key (realm_id, securable_catalog_id, securable_id, grantee_catalog_id, grantee_id, privilege_code)=(POLARIS, 0, 920353451338950398, 920353451338950398, 27460542378657136, 32) already exists.' (error code 0, sql-state '23505'), after 1 attempts and 5000 milliseconds

[2026-06-26T02:46:05.800] ERROR | org.apache.polaris.service.exception.IcebergExceptionMapper
  Unhandled exception returning INTERNAL_SERVER_ERROR

[2026-06-26T02:46:05.896] INFO | org.apache.polaris.service.exception.IcebergExceptionMapper
  Ha

In [11]:
# The exception stack trace — search broadly around the failure timestamps,
# including DEBUG level, for exception class names
hits = search_logs_raw({
    "query": {"bool": {
        "must": [{"range": {"@timestamp": {"gte": "now-15m"}}},
                 {"term": {"kubernetes.container_name.keyword": "benchmarks-polaris"}}],
        "should": [
            {"wildcard": {"message": "*Exception*"}},
            {"wildcard": {"message": "*Caused by*"}},
            {"wildcard": {"message": "*Access*Denied*"}},
            {"wildcard": {"message": "*403*"}},
            {"wildcard": {"message": "*credential*"}},
            {"wildcard": {"message": "*not authorized*"}},
            {"wildcard": {"message": "*write*"}},
            {"match_phrase": {"loggerName": "TableCleanupTaskHandler"}}],
        "minimum_should_match": 1}},
    "sort": [{"@timestamp": {"order": "asc"}}]
}, size=60)
print(f"Found {len(hits)}:\n")
for h in hits:
    s = h["_source"]
    print(f"[{s.get('@timestamp','')[:23]}] {s.get('level')} | {s.get('loggerName','')}")
    print(f"  {s.get('message','')}\n")

Found 60:

[2026-06-26T02:41:36.865] DEBUG | org.apache.polaris.service.catalog.io.StorageAccessConfigProvider
  tableIdentifier=vp-ns.C1-e1a1a7 tableLocation=[s3a://data-catalog-bucket/view-test-nogrant/vp-ns/C1-e1a1a7/] Fetching client credentials for table

[2026-06-26T02:41:36.865] DEBUG | org.apache.polaris.core.storage.cache.StorageCredentialCache
  key=StorageCredentialCacheKey{realmId=POLARIS, catalogId=0, storageConfigSerializedStr={"@type":"AwsStorageConfigurationInfo","allowedLocations":["s3a://data-catalog-bucket/view-test-nogrant/","s3a://data-catalog-bucket/"],"endpoint":"http://192.168.139.2:9000","endpointInternal":"http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000","pathStyleAccess":true,"storageType":"S3","fileIoImplClassName":"org.apache.iceberg.aws.s3.S3FileIO"}, allowedListAction=false, allowedReadLocations=[s3a://data-catalog-bucket/view-test-nogrant/vp-ns/C1-e1a1a7/], allowedWriteLocations=[s3a://data-catalog-bucket/view-test-nogrant/vp-ns/C1-e1a1a7/]}

In [34]:
# Generate fresh cleanup activity, THEN query (run this first if needed)
import uuid, time
TBL = f"logtest-{uuid.uuid4().hex[:8]}"
# create table + write a file (reuse your fs + creds), then:
requests.delete(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/tables/{TBL}?purgeRequested=true", headers=h(tok))
time.sleep(5)
# now run the 3 log queries above

In [6]:
# Create catalog with purge-view-metadata-on-drop = TRUE (the DEFAULT we're testing)
print("Create catalog (purge-view-metadata-on-drop = TRUE)")
r = requests.post(f"{BASE_MGMT}/catalogs", headers=h(tok), json={
    "catalog": {
        "name": TEST_CATALOG,
        "type": "INTERNAL",
        "properties": {
            "default-base-location": TEST_LOCATION,
            "polaris.config.purge-view-metadata-on-drop": "true",  # explicit, = default
        },
        "storageConfigInfo": {
            "storageType": "S3",
            "allowedLocations": [TEST_LOCATION],
            "pathStyleAccess": True,
            "endpoint": "http://192.168.139.2:9000",
            "endpointInternal": "http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000",
        },
    }
})
print_response(r, "Create catalog")

# Grant content management so we have view privileges
requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}/catalog-roles/catalog_admin/grants",
             headers=h(tok), json={"grant": {"type":"catalog","privilege":"CATALOG_MANAGE_CONTENT"}})

# Namespace
requests.post(f"{BASE_CAT}/{TEST_CATALOG}/namespaces", headers=h(tok),
              json={"namespace": [TEST_NS], "properties": {}})
print("✅ catalog + namespace ready")


Create catalog (purge-view-metadata-on-drop = TRUE)

Create catalog:
  Status:     409
  Request-Id: 626665e0-9248-4452-921e-0535b85a9a56_0000000000000000955
  Body: {
  "error": {
    "message": "Cannot create Catalog view-purge-test. Catalog already exists or resolution failed",
    "type": "AlreadyExistsException",
    "code": 409
  }
}
✅ catalog + namespace ready


## Case 1 — Drop view with purge-view-metadata-on-drop = TRUE

In [ ]:
# Create a view, confirm its metadata file exists in MinIO, then drop it
VIEW1 = "view-true"
print(f"Create view: {VIEW1}")
r = requests.post(
    f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/views",
    headers=h(tok),
    json={
        "name": VIEW1,
        "default-namespace": [TEST_NS],
        "schema": {"type": "struct", "schema-id": 0,
                   "fields": [{"id": 1, "name": "id", "type": "long", "required": True}]},
        "view-version": {
            "version-id": 1, "timestamp-ms": int(time.time()*1000), "schema-id": 0,
            "default-namespace": [TEST_NS], "summary": {"engine": "spark"},
            "representations": [{"type": "sql", "sql": "SELECT 1 AS id", "dialect": "spark"}],
        },
        "location": f"{TEST_LOCATION}{TEST_NS}/{VIEW1}/",
    }
)
print_response(r, "Create view")


In [ ]:
# Confirm the view's metadata file exists in MinIO BEFORE drop
print("MinIO files for view BEFORE drop:")
files_before = list_view_files(VIEW1)
for f in files_before:
    print(f"  {f}")
print(f"  total: {len(files_before)} file(s)")


In [ ]:
# Drop the view — purge-view-metadata-on-drop is TRUE
# Question: does this succeed? does it delete the MinIO metadata file?
print(f"Drop view {VIEW1} (no purgeRequested flag)")
r = requests.delete(
    f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/views/{VIEW1}",
    headers=h(tok)
)
print_response(r, "Drop view (config=true)")
print(f"\nDrop status: {r.status_code}")


In [ ]:
# Confirm MinIO state AFTER drop
import time as _t
_t.sleep(2)  # allow async cleanup if any
print("MinIO files for view AFTER drop:")
files_after = list_view_files(VIEW1)
for f in files_after:
    print(f"  {f}")
print(f"  total: {len(files_after)} file(s)")
print()
if len(files_before) > 0 and len(files_after) == 0:
    print("✅ RESULT: metadata file WAS deleted from MinIO (purge-view-metadata=true worked)")
elif len(files_before) > 0 and len(files_after) > 0:
    print("ℹ️  RESULT: metadata file REMAINS in MinIO despite config=true")
elif len(files_before) == 0:
    print("ℹ️  No metadata file was created in MinIO to begin with")


## Case 2 — Drop view with purge-view-metadata-on-drop = FALSE

In [ ]:
# Switch catalog config to FALSE
print("Set purge-view-metadata-on-drop = FALSE")
cat = requests.get(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok)).json()
merged = {**cat.get("properties", {}), "polaris.config.purge-view-metadata-on-drop": "false"}
r = requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok),
                 json={"currentEntityVersion": cat["entityVersion"], "properties": merged})
print(f"  update: {r.status_code}")
print(f"  config now: {r.json().get('properties', {}).get('polaris.config.purge-view-metadata-on-drop')}")


In [ ]:
# Create a second view, confirm metadata exists, drop it
VIEW2 = "view-false"
print(f"Create view: {VIEW2}")
requests.post(
    f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/views",
    headers=h(tok),
    json={
        "name": VIEW2,
        "default-namespace": [TEST_NS],
        "schema": {"type": "struct", "schema-id": 0,
                   "fields": [{"id": 1, "name": "id", "type": "long", "required": True}]},
        "view-version": {
            "version-id": 1, "timestamp-ms": int(time.time()*1000), "schema-id": 0,
            "default-namespace": [TEST_NS], "summary": {"engine": "spark"},
            "representations": [{"type": "sql", "sql": "SELECT 1 AS id", "dialect": "spark"}],
        },
        "location": f"{TEST_LOCATION}{TEST_NS}/{VIEW2}/",
    }
)

files_before2 = list_view_files(VIEW2)
print(f"MinIO files BEFORE drop: {len(files_before2)}")
for f in files_before2: print(f"  {f}")


In [ ]:
# Drop view with config=false
print(f"Drop view {VIEW2}")
r = requests.delete(
    f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/views/{VIEW2}",
    headers=h(tok)
)
print_response(r, "Drop view (config=false)")

import time as _t
_t.sleep(2)
files_after2 = list_view_files(VIEW2)
print(f"\nMinIO files AFTER drop: {len(files_after2)}")
for f in files_after2: print(f"  {f}")
print()
if len(files_before2) > 0 and len(files_after2) > 0:
    print("✅ RESULT: metadata file REMAINS (config=false → metadata-only drop)")
elif len(files_before2) > 0 and len(files_after2) == 0:
    print("ℹ️  RESULT: metadata file deleted despite config=false")
else:
    print("ℹ️  No metadata file created to begin with")


## Summary of findings

In [ ]:
# Print comparison table
print("="*60)
print("VIEW PURGE BEHAVIOR — TEST RESULTS")
print("="*60)
print()
print(f"Case 1: purge-view-metadata-on-drop = TRUE")
print(f"  drop status:        {r.status_code if False else 'see Case 1 cell'}")
print(f"  MinIO before/after: {len(files_before)} → {len(files_after)}")
print()
print(f"Case 2: purge-view-metadata-on-drop = FALSE")
print(f"  MinIO before/after: {len(files_before2)} → {len(files_after2)}")
print()
print("Interpretation:")
print("  TRUE  → Polaris deletes the view metadata file from MinIO on drop")
print("  FALSE → Polaris leaves the metadata file; only PostgreSQL record removed")
print()
print("Note: views hold only a small metadata file (the query definition),")
print("      never Parquet data files. So this only affects that one small file.")


## Data-file proof — what does purgeRequested=true actually remove?

Earlier tests used tables/views that may have had only metadata files.
This test writes a REAL Parquet data file, commits it, then purges —
counting **data files** and **metadata files** separately, before and after.

This proves precisely what `purgeRequested=true` removes vs leaves in MinIO
in this environment.


In [ ]:
# Write a real Parquet data file + commit, then purge, count separately
import pyarrow as pa, pyarrow.parquet as pq, uuid, io, time

# Self-contained config helper (set both purge flags on the catalog)
def set_config(purge_view, drop_purge):
    cat = requests.get(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok)).json()
    merged = {**cat.get("properties", {}),
              "polaris.config.purge-view-metadata-on-drop": purge_view,
              "polaris.config.drop-with-purge.enabled": drop_purge}
    requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok),
                 json={"currentEntityVersion": cat["entityVersion"], "properties": merged})

# Ensure purge enabled + granted
set_config("false", "true")
requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}/catalog-roles/catalog_admin/grants",
             headers=h(tok), json={"grant":{"type":"catalog","privilege":"CATALOG_MANAGE_CONTENT"}})

TBL = f"datatbl-{uuid.uuid4().hex[:8]}"
tbl_loc = f"{TEST_LOCATION}{TEST_NS}/{TBL}"

# Create table
r = requests.post(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/tables", headers=h(tok),
    json={"name": TBL, "location": f"{tbl_loc}/",
          "schema": {"type":"struct","fields":[
              {"id":1,"name":"id","type":"long","required":True},
              {"id":2,"name":"val","type":"string","required":False}]},
          "properties": {"write.format.default":"parquet"}})
print(f"Create table: {r.status_code}")

# Write Parquet data file directly to MinIO
data = pa.table({"id": pa.array([1,2,3], type=pa.int64()),
                 "val": pa.array(["a","b","c"], type=pa.string())})
pq_path = f"{tbl_loc}/data/{uuid.uuid4().hex}.parquet"
with fs.open(pq_path.replace("s3a://","s3://"), "wb") as f:
    pq.write_table(data, f)
FILE_SIZE = fs.info(pq_path.replace("s3a://","s3://"))["size"]
print(f"Wrote Parquet: {FILE_SIZE} bytes, 3 rows")

def count_files(table_name):
    base = f"data-catalog-bucket/view-purge-test/{TEST_NS}/{table_name}/"
    try:
        allf = [f for f in fs.find(base) if fs.info(f)["size"] > 0]
    except FileNotFoundError:
        return 0, 0, []
    data_files = [f for f in allf if "/data/" in f]
    meta_files = [f for f in allf if "/metadata/" in f]
    return len(data_files), len(meta_files), allf

d_before, m_before, _ = count_files(TBL)
print(f"\nBEFORE purge:")
print(f"  data files (.parquet): {d_before}")
print(f"  metadata files:        {m_before}")

# Drop WITH purge
r = requests.delete(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/tables/{TBL}?purgeRequested=true", headers=h(tok))
print(f"\nDrop status: {r.status_code}")
time.sleep(3)

d_after, m_after, _ = count_files(TBL)
print(f"\nAFTER purge:")
print(f"  data files (.parquet): {d_after}")
print(f"  metadata files:        {m_after}")
print()
print("="*60)
print("RESULT — what purgeRequested=true removed from MinIO:")
print(f"  data files:     {d_before} -> {d_after}  {'DELETED' if d_before>0 and d_after==0 else 'ORPHANED'}")
print(f"  metadata files: {m_before} -> {m_after}  {'DELETED' if m_before>0 and m_after==0 else 'ORPHANED'}")
print("="*60)


## Re-check — async cleanup verification

**Critical insight (from Polaris docs + GitHub issues #289, #1195, #1448):**
Polaris purge is **asynchronous**. The DELETE returns 204 immediately, but the
actual file deletion runs later in a background task (`TableCleanupTaskHandler`).

Our earlier tests waited only ~3 seconds — likely TOO EARLY to see the async
cleanup complete. These cells:
  1. Re-check earlier-purged tables NOW (minutes later)
  2. Run a fresh purge and poll over a longer window
  3. Pull the actual Polaris cleanup-task logs from OpenSearch


In [ ]:
# 1. Re-check files for tables we purged earlier in this session.
#    If GONE now → async cleanup worked, we just measured too early.
import s3fs

# Adjust these to the table names you purged earlier (from cell outputs)
earlier_purged = [
    # e.g. "datatbl-xxxxxxxx", "writetest-5cd8cebe", "t-xxxxxxxx"
    # fill in any names you saw purged earlier, or leave to auto-scan below
]

# Auto-scan: list everything still under the test namespace
print("Scanning ALL remaining files under the test namespace...")
base = f"data-catalog-bucket/view-purge-test/vp-ns/"
try:
    all_remaining = [f for f in fs.find(base) if fs.info(f)["size"] > 0]
    # group by table/view name
    from collections import defaultdict
    by_entity = defaultdict(list)
    for f in all_remaining:
        # path: data-catalog-bucket/view-purge-test/vp-ns/<entity>/...
        parts = f.split("/vp-ns/")[1].split("/")
        by_entity[parts[0]].append(f)
    print(f"\nEntities with files still in MinIO: {len(by_entity)}")
    for ent, files in sorted(by_entity.items()):
        print(f"  {ent}: {len(files)} file(s)")
except FileNotFoundError:
    print("  namespace prefix empty — everything cleaned up ✅")


In [ ]:
# 2. Fresh purge with LONGER polling window (async cleanup may take time)
import pyarrow as pa, pyarrow.parquet as pq, uuid, time

def set_config(pv, dp):
    cat = requests.get(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok)).json()
    merged = {**cat.get("properties", {}),
              "polaris.config.purge-view-metadata-on-drop": pv,
              "polaris.config.drop-with-purge.enabled": dp}
    requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok),
                 json={"currentEntityVersion": cat["entityVersion"], "properties": merged})

set_config("false", "true")
requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}/catalog-roles/catalog_admin/grants",
             headers=h(tok), json={"grant":{"type":"catalog","privilege":"CATALOG_MANAGE_CONTENT"}})

TBL = f"async-{uuid.uuid4().hex[:8]}"
loc = f"{TEST_LOCATION}{TEST_NS}/{TBL}"
requests.post(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/tables", headers=h(tok),
    json={"name": TBL, "location": f"{loc}/",
          "schema": {"type":"struct","fields":[{"id":1,"name":"id","type":"long","required":True}]},
          "properties": {"write.format.default":"parquet"}})

# Write Parquet
data = pa.table({"id": pa.array([1,2,3], type=pa.int64())})
pqp = f"{loc}/data/{uuid.uuid4().hex}.parquet"
with fs.open(pqp.replace("s3a://","s3://"), "wb") as f:
    pq.write_table(data, f)

def count(name):
    b = f"data-catalog-bucket/view-purge-test/{TEST_NS}/{name}/"
    try:
        return len([f for f in fs.find(b) if fs.info(f)["size"] > 0])
    except FileNotFoundError:
        return 0

print(f"Table: {TBL}")
print(f"Files before purge: {count(TBL)}")

# Purge
r = requests.delete(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/tables/{TBL}?purgeRequested=true", headers=h(tok))
print(f"Drop status: {r.status_code}")

# Poll over 60 seconds
print("\nPolling MinIO for async cleanup:")
for t in [2, 5, 10, 20, 30, 45, 60]:
    time.sleep(t - (0 if t == 2 else [2,5,10,20,30,45,60][[2,5,10,20,30,45,60].index(t)-1]))
    n = count(TBL)
    print(f"  +{t:2}s: {n} file(s) {'→ CLEANED ✅' if n == 0 else ''}")
    if n == 0:
        print("\n✅ Async cleanup COMPLETED — files deleted by background task")
        break
else:
    print("\n⚠️  Files still present after 60s — cleanup task may have failed")
    print("    → check Polaris logs (next cell) for TableCleanupTaskHandler errors")


## 3. Polaris cleanup-task logs (OpenSearch)

The definitive answer lives in the Polaris pod logs. The cleanup task logs as
`TableCleanupTaskHandler` / `TaskExecutorImpl`. This queries the company
OpenSearch for those entries around our purge operations.

If the task ran and SUCCEEDED → "Task successfully handled"
If it FAILED (like GitHub #1195) → an exception will appear


In [ ]:
# Query OpenSearch for Polaris cleanup-task logs
# Adjust index name if the company log index differs (e.g. kube-fb-log)
from opensearchpy import OpenSearch

# Reuse company OpenSearch connection from utils if available, else define here
try:
    client = os_client()
    INDEX = LOG_INDEX
except Exception:
    # fallback — adjust to your environment
    client = OpenSearch(
        hosts=[{"host": "192.168.194.1", "port": 9200}],
        http_auth=("admin", "Str0ngP@ssw0rd123!"),
        use_ssl=True, verify_certs=False, ssl_show_warn=False)
    INDEX = "kube-fb-log"

# Search for cleanup task activity in the last 15 minutes
resp = client.search(index=INDEX, body={
    "query": {
        "bool": {
            "must": [
                {"range": {"@timestamp": {"gte": "now-15m"}}},
            ],
            "should": [
                {"match_phrase": {"message": "TableCleanupTaskHandler"}},
                {"match_phrase": {"message": "TaskExecutorImpl"}},
                {"match_phrase": {"message": "cleanup task"}},
                {"match_phrase": {"message": "Handling task"}},
                {"match_phrase": {"message": "purge"}},
            ],
            "minimum_should_match": 1
        }
    },
    "sort": [{"@timestamp": {"order": "desc"}}],
    "size": 30
})

hits = resp["hits"]["hits"]
print(f"Found {len(hits)} cleanup-task log entries (last 15 min):\n")
for hit in hits:
    s = hit["_source"]
    ts = s.get("@timestamp", "")[:19]
    lvl = s.get("level", s.get("level.keyword", "?"))
    msg = s.get("message", "")[:200]
    print(f"[{ts}] {lvl}")
    print(f"  {msg}")
    print()

if not hits:
    print("No cleanup-task logs found. Possible reasons:")
    print("  → Polaris logs not JSON-parsed in OpenSearch (message field is plain text)")
    print("  → try a broader query: just match 'task' or 'cleanup' in message")
    print("  → confirm the index name (kube-fb-log vs k8s-logs-*)")


## Diagnostic — Why are files orphaned? (for System Engineer)

We proved Polaris removes the PostgreSQL record but does NOT delete the MinIO
file (both tables and views, 1→1). The cell below prints the exact commands a
**System Engineer** should run (with kubectl access, OUTSIDE this notebook) to
determine the root cause.

The notebook itself has no kubectl access, so these are printed as guidance.


In [ ]:
# Prints diagnostic commands for the System Engineer to run with kubectl.
# This notebook cannot run kubectl — these are guidance only.

cat = requests.get(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok)).json()
sc = cat.get("storageConfigInfo", {})
ep_internal = sc.get("endpointInternal", "(not set)")
ep_external = sc.get("endpoint", "(not set)")

print("="*70)
print("DIAGNOSTIC: Polaris → MinIO file deletion failure")
print("="*70)
print()
print("PROVEN (from this notebook):")
print("  • Polaris removes PostgreSQL records on drop/purge (204)")
print("  • Polaris does NOT delete MinIO files (orphaned, 1 → 1)")
print("  • Affects BOTH tables and views")
print()
print(f"Storage endpoint Polaris uses server-side (endpointInternal):")
print(f"  {ep_internal}")
print(f"External endpoint (query engines):")
print(f"  {ep_external}")
print()
print("-"*70)
print("CANDIDATE A — Polaris pod cannot REACH MinIO at endpointInternal")
print("-"*70)
print("System Engineer, run from a shell with kubectl access:")
print()
print("  # 1. Find the Polaris pod")
print("  kubectl get pods -n datahub-hynix | grep polaris")
print()
print("  # 2. Test DNS resolution + reachability from INSIDE the Polaris pod")
print(f"  kubectl exec -n datahub-hynix <polaris-pod> -- \\")
print(f"    curl -v {ep_internal}/minio/health/live")
print()
print("  Expected if healthy: HTTP 200")
print("  If 'Could not resolve host' → DNS issue (Candidate A confirmed)")
print("  If 'Connection refused/timeout' → network/service issue")
print()
print("-"*70)
print("CANDIDATE B — Polaris storage CREDENTIALS lack delete permission")
print("-"*70)
print("System Engineer:")
print()
print("  # Check what credentials Polaris is configured with")
print("  kubectl get secret -n datahub-hynix | grep -i polaris")
print("  kubectl describe deployment -n datahub-hynix <polaris-deploy> | grep -i -A2 'AWS\\|S3\\|MINIO\\|CRED'")
print()
print("  # Verify those creds can delete from MinIO (test with mc or aws cli):")
print("  #   - if creds can LIST/GET but not DELETE → Candidate B confirmed")
print("  #   - if creds are the static MinIO root key → should have delete perms")
print()
print("-"*70)
print("CORRELATION CHECK")
print("-"*70)
print("  At catalog creation earlier in testing, we saw: UnknownHostException")
print("  → that is a DNS resolution failure")
print("  → strongly points to Candidate A (endpointInternal not resolvable")
print("    from the Polaris pod)")
print()
print("  If Candidate A: fix endpointInternal so the Polaris pod can resolve")
print("  + reach MinIO (correct K8s service name / namespace / port).")
print("="*70)


In [ ]:
# Final cleanup
print("Final cleanup...")
cat = requests.get(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok)).json()
merged = {**cat.get("properties", {}),
          "polaris.config.drop-with-purge.enabled": "true",
          "polaris.config.purge-view-metadata-on-drop": "false"}
requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok),
             json={"currentEntityVersion": cat["entityVersion"], "properties": merged})
for ns in requests.get(f"{BASE_CAT}/{TEST_CATALOG}/namespaces", headers=h(tok)).json().get("namespaces", []):
    nsn = ns[0] if isinstance(ns, list) else ns
    for v in requests.get(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", []):
        requests.delete(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{nsn}/views/{v['name']}?purgeRequested=true", headers=h(tok))
    requests.delete(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{nsn}", headers=h(tok))
requests.delete(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}?purgeRequested=true", headers=h(tok))
print("✅ test catalog removed")
